In [1]:
import requests
import time
import requests
import pymssql
import calendar
import re
import pandas as pd
from selenium import webdriver
from selenium.webdriver.edge.options import Options
from selenium.webdriver.chrome.service import Service
from selenium.common.exceptions import TimeoutException
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import Select
from bs4 import BeautifulSoup
from datetime import datetime, timedelta


# 資料庫設定
db_settings = {
    "host": "127.0.0.1",
    "user": "user",
    "password": "0000",
    "database": "NCU_database",
    "charset": "utf8"
}

# 儲存台灣50前10的陣列
taiwan50 = []

# 記錄上次插入的股票數據
last_record = {}

start_date = datetime(2022, 1, 1).date()
end_date = datetime.now().date()
current_date = start_date

dates = []
while current_date <= end_date:
    dates.append(current_date.strftime('%Y%m%d'))
    # 增加一個月
    current_date = datetime(current_date.year + (current_date.month // 12), 
                          (current_date.month % 12) + 1, 
                          1).date()
print(dates)

['20220101', '20220201', '20220301', '20220401', '20220501', '20220601', '20220701', '20220801', '20220901', '20221001', '20221101', '20221201', '20230101', '20230201', '20230301', '20230401', '20230501', '20230601', '20230701', '20230801', '20230901', '20231001', '20231101', '20231201', '20240101', '20240201', '20240301', '20240401', '20240501', '20240601', '20240701', '20240801', '20240901', '20241001', '20241101', '20241201', '20250101', '20250201', '20250301']


In [2]:
def find_Taiwan50():
    options = Options()
    options.add_argument("--headless")  # 執行時不顯示瀏覽器
    options.add_argument("--disable-notifications")  # 禁止瀏覽器的彈跳通知
    driver = webdriver.Edge(options=options)
    driver.get("https://www.cmoney.tw/etf/tw/0050/fundholding")
        
    # TODO : 練習2
    time.sleep(3)
    html_list = driver.find_elements(By.CSS_SELECTOR, 'div[class="cm-table"] tbody tr')
    
    taiwan50.clear()
    
    counter = 0
    for html in html_list:
        counter += 1
        if counter > 10:
            break
        html = html.find_elements(By.CSS_SELECTOR, 'td')
        taiwan50.append(html[0].text)
        
    
    print (f"The Top {len(taiwan50)} stocks:\n{taiwan50}\n")
    driver.quit()

def fetch_stock_data(stock_code, date):
    """ 從 API 抓取指定股票的數據 """
    # url
    url = f"https://www.twse.com.tw/exchangeReport/STOCK_DAY?response=json&date={date}&stockNo={stock_code}"
 
    try:
        response = requests.get(url)
        response.raise_for_status()  # 檢查 HTTP 錯誤
        data = response.json()
        
        # 檢查數據結構並提取欄位名稱和數據
        if 'fields' not in data or 'data' not in data:
            print(f"❌ API 回傳數據格式不正確: {data}")
            return None
        
        fields = data['fields']  # 欄位名稱
        data = data['data']      # 數據內容
        content = {"fields": fields, "data": data}
        
        return content
    except Exception as e:
        print(f"❌ 無法獲取 {stock_code} 的數據: {e}")
    
    return None

def parse_stock_data(stock_data):
    """ 解析 API 數據，並確保所有 `REAL` 類型數據為 float """
    def safe_float(value):
        """ 將字串轉換為浮點數，若為 'X' 則回傳 0.0 """
        try:
            return float(value.replace(",", ""))
        except ValueError:
            print(f"Fix {stock_data[0]} data {value} " )
            return 0.0
    
    
    try:
        # 民國轉西元
        ce = stock_data[0].split("/")
        ce[0] = str(int(ce[0]) + 1911)
        ce = "-".join(ce)
        
        return {
            "date": ce,                                             # 交易日期
            "time": "00:00:00",                                     # 交易時間 (因為沒有確切的時間)
            "trade_volume": int(stock_data[1].replace(",", "")),    # 成交股數
            "trade_value": int (stock_data[2].replace(",", "")),    # 成交金額
            "open_price": safe_float(stock_data[3]),                # 開盤價
            "high_price": safe_float(stock_data[4]),                # 最高價
            "low_price": safe_float(stock_data[5]),                 # 最低價
            "close_price": safe_float(stock_data[6].replace(",", "")),   # 收盤價
            "price_change": safe_float(stock_data[7]),              # 漲跌價差
            "trade_count": int(stock_data[8].replace(",", ""))      # 成交筆數
        }
        
    except Exception as e:
        print(f"❌ 無法獲取 {ce} 的數據: {e}")
        return None
    
def collect_data(date_list, stock_code):
    conn = pymssql.connect(**db_settings)
    cursor = conn.cursor()
    
    
    
    for date in date_list:
        record = fetch_stock_data(stock_code, date)
        
        for data in record["data"]:
            parsed_data = parse_stock_data(data)
            
            stored_data = (
                stock_code,
                parsed_data["date"],            # 日期
                parsed_data["time"],            # 交易時間
                parsed_data["trade_volume"],    # 成交股數
                parsed_data["trade_value"],     # 成交金額
                parsed_data["open_price"],      # 開盤價
                parsed_data["high_price"],      # 最高價
                parsed_data["low_price"],       # 最低價
                parsed_data["close_price"],     # 收盤價
                parsed_data["price_change"],    # 漲跌價差
                parsed_data["trade_count"]      # 成交筆數
            )
            
            print(stored_data)
             # SQL 插入語句
            insert_query = """
                INSERT INTO dbo.stock_price_info
                (stock_code, date, time, tv, t, o, h, l, c, d, v)
                VALUES
                (%s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s)
            """
            cursor.execute(insert_query, stored_data)
            conn.commit()
    
    conn.close()
    
    
if __name__ == "__main__":
    find_Taiwan50()
    
    for stock_code in taiwan50:
        print(f"Stock code: {stock_code}")
        collect_data(dates, stock_code)
    
        

The Top 10 stocks:
['2330', '2891', '2883', '2884', '2317', '2890', '2886', '2887', '2303', '2885']

Stock code: 2330
('2330', '2022-01-03', '00:00:00', 73703302, 46249716919, 619.0, 632.0, 618.0, 631.0, 16.0, 88508)
('2330', '2022-01-04', '00:00:00', 90945643, 59188199534, 645.0, 656.0, 644.0, 656.0, 25.0, 106409)
('2330', '2022-01-05', '00:00:00', 72505550, 47582832784, 669.0, 669.0, 646.0, 650.0, -6.0, 64712)
('2330', '2022-01-06', '00:00:00', 57490736, 36817638522, 638.0, 646.0, 636.0, 644.0, -6.0, 53430)
('2330', '2022-01-07', '00:00:00', 39847766, 25358237656, 643.0, 646.0, 632.0, 634.0, -10.0, 44497)
('2330', '2022-01-10', '00:00:00', 39286754, 25052138297, 628.0, 645.0, 627.0, 643.0, 9.0, 33453)
('2330', '2022-01-11', '00:00:00', 36186544, 23361588355, 646.0, 651.0, 639.0, 651.0, 8.0, 26849)
('2330', '2022-01-12', '00:00:00', 40832298, 26762357054, 657.0, 660.0, 650.0, 660.0, 9.0, 39164)
('2330', '2022-01-13', '00:00:00', 39172028, 25813559647, 658.0, 662.0, 655.0, 661.0, 1.0, 